# SER Dataset Summary + Combined Manifest (6-class)

Aggregates the label CSVs produced by `predownload_to_drive.ipynb` (academic datasets) and
`colab_qwen_emotion_label.ipynb` (`qwen_911`) into one training manifest, and reports the
6-class distribution.

- Reads `CLEAR/emotion_data/labels/*_labels.csv` (relative paths, one row per clip).
- Writes `labels/all_6class_labels.csv` = concatenation of every `*_labels.csv`.
- `scanner_domain_manifest.csv` is intentionally NOT matched (it's `*_manifest`, not `*_labels`),
  so scanner/dispatch audio never enters the caller-emotion corpus.
- Audio lives in `CLEAR/emotion_data/zips/<dataset>.zip`; extract locally when training.


In [ ]:
# Cell 1: Mount + paths
import csv
from pathlib import Path
from collections import Counter, defaultdict
from google.colab import drive
drive.mount('/content/drive')

DATA = Path('/content/drive/MyDrive/CLEAR/emotion_data')
LABELS_DIR = DATA / 'labels'
ZIPS_DIR = DATA / 'zips'
CLASS6_ORDER = ["panic", "fear", "urgency", "distress", "confusion", "neutral"]
print(f'Labels: {LABELS_DIR}')
print(f'Zips  : {ZIPS_DIR}')


In [ ]:
# Cell 2: Aggregate all *_labels.csv -> combined manifest + distribution report
label_files = sorted(LABELS_DIR.glob('*_labels.csv'))   # excludes *_manifest.csv (scanner)
if not label_files:
    print('No *_labels.csv found. Run predownload_to_drive.ipynb (+ the Qwen notebook) first.')
else:
    all_rows = []
    per_source = Counter()
    per_source_class = defaultdict(Counter)
    fields = ["path", "label", "class_6", "source", "speaker_id"]
    for lf in label_files:
        with open(lf) as f:
            rows = list(csv.DictReader(f))
        for r in rows:
            all_rows.append(r)
            per_source[r['source']] += 1
            per_source_class[r['source']][r['class_6']] += 1

    out = LABELS_DIR / 'all_6class_labels.csv'
    with open(out, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
        for r in all_rows: w.writerow({k: r.get(k, '') for k in fields})

    print(f'Combined manifest: {len(all_rows)} clips -> {out}\n')
    print(f'{"source":>18} | {"total":>6} | ' + ' '.join(f'{c[:5]:>6}' for c in CLASS6_ORDER))
    print('-' * 78)
    for s in sorted(per_source):
        cc = per_source_class[s]
        print(f'{s:>18} | {per_source[s]:>6} | ' + ' '.join(f'{cc.get(c,0):>6}' for c in CLASS6_ORDER))
    total = Counter(r['class_6'] for r in all_rows)
    print('-' * 78)
    print(f'{"TOTAL":>18} | {len(all_rows):>6} | ' + ' '.join(f'{total.get(c,0):>6}' for c in CLASS6_ORDER))
    print('\nClass balance (%):')
    for c in CLASS6_ORDER:
        n = total.get(c, 0)
        print(f'  {c:>10}: {n:>6}  ({100*n/max(1,len(all_rows)):.1f}%)')


In [ ]:
# Cell 3: Zip inventory on Drive
print('Audio zips:')
tot = 0.0
for z in sorted(ZIPS_DIR.glob('*.zip')):
    gb = z.stat().st_size / 1024**3; tot += gb
    print(f'  {z.name:>24}: {gb:.2f} GB')
print(f'  {"TOTAL":>24}: {tot:.2f} GB')
manifest = LABELS_DIR / 'scanner_domain_manifest.csv'
if manifest.exists():
    with open(manifest) as f: n = sum(1 for _ in f) - 1
    print(f'\nscanner_domain_manifest.csv: {max(0,n)} rows (separate — domain fine-tuning, not in the 6-class corpus)')
